In [ ]:
import os
import joblib
import pandas as pd

from pathlib import Path
from scipy.sparse import hstack, csr_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split


# ============================================================
# PROJECT
# ============================================================

PROJECT_PATH = Path(
    r"C:\Users\Rajas\Downloads\youtube-project"
)


DATA_PATH = PROJECT_PATH / "data" / "youtube_data.csv"
MODEL_DIR = PROJECT_PATH / "models"

MODEL_DIR.mkdir(exist_ok=True)

print("Project:", PROJECT_PATH)
print("Dataset exists:", DATA_PATH.exists())


# ============================================================
# MODEL FILES
# ============================================================

MODEL_PATH = MODEL_DIR / "best_model.pkl"
TITLE_PATH = MODEL_DIR / "tfidf_title.pkl"
DESCRIPTION_PATH = MODEL_DIR / "tfidf_description.pkl"
TAGS_PATH = MODEL_DIR / "tfidf_tags.pkl"


# ============================================================
# LOAD EXISTING MODEL IF AVAILABLE
# ============================================================

if all([
    MODEL_PATH.exists(),
    TITLE_PATH.exists(),
    DESCRIPTION_PATH.exists(),
    TAGS_PATH.exists()
]):

    print("\n✅ Existing model found.")
    print("Loading...")

    best_model = joblib.load(MODEL_PATH)
    tfidf_title = joblib.load(TITLE_PATH)
    tfidf_description = joblib.load(DESCRIPTION_PATH)
    tfidf_tags = joblib.load(TAGS_PATH)

    print("✅ Model loaded successfully!")


# ============================================================
# OTHERWISE TRAIN NEW MODEL
# ============================================================

else:

    print("\n⚠️ Model files not found.")
    print("Training a new Logistic Regression model...\n")

    if not DATA_PATH.exists():

        raise FileNotFoundError(
            f"youtube_data.csv was not found at:\n{DATA_PATH}"
        )

    df = pd.read_csv(DATA_PATH)

    print("Dataset:", df.shape)


    # --------------------------------------------------------
    # CLEAN TEXT
    # --------------------------------------------------------

    for column in [
        "title",
        "description",
        "hashtags"
    ]:

        df[column] = (
            df[column]
            .fillna("")
            .astype(str)
        )


    # --------------------------------------------------------
    # VIEWS
    # --------------------------------------------------------

    df["views"] = pd.to_numeric(
        df["views"],
        errors="coerce"
    ).fillna(0)


    # --------------------------------------------------------
    # TOP 10% HIGH-VIEW LABEL
    # --------------------------------------------------------

    threshold = df["views"].quantile(0.90)

    df["Trending"] = (
        df["views"] >= threshold
    ).astype(int)

    print(
        "High-view threshold:",
        round(threshold)
    )

    print(
        "High-view videos:",
        df["Trending"].sum()
    )


    # --------------------------------------------------------
    # NUMERIC FEATURES
    # --------------------------------------------------------

    df["title_length"] = df["title"].str.len()

    df["description_length"] = (
        df["description"].str.len()
    )

    df["description_word_count"] = (
        df["description"]
        .str.split()
        .str.len()
    )

    df["tag_count"] = (
        df["hashtags"]
        .str.split()
        .str.len()
    )

    df["has_description"] = (
        df["description"].str.strip() != ""
    ).astype(int)

    df["has_tags"] = (
        df["hashtags"].str.strip() != ""
    ).astype(int)

    df["title_has_question"] = (
        df["title"].str.contains(
            r"\?",
            regex=True
        )
    ).astype(int)


    # --------------------------------------------------------
    # TRAIN / TEST SPLIT
    # --------------------------------------------------------

    train_idx, test_idx = train_test_split(
        df.index,
        test_size=0.20,
        random_state=42,
        stratify=df["Trending"]
    )


    # --------------------------------------------------------
    # TF-IDF
    # --------------------------------------------------------

    tfidf_title = TfidfVectorizer(
        max_features=5000,
        stop_words="english",
        ngram_range=(1, 2)
    )

    tfidf_description = TfidfVectorizer(
        max_features=5000,
        stop_words="english",
        ngram_range=(1, 2)
    )

    tfidf_tags = TfidfVectorizer(
        max_features=3000
    )


    title_train = tfidf_title.fit_transform(
        df.loc[train_idx, "title"]
    )

    description_train = (
        tfidf_description.fit_transform(
            df.loc[train_idx, "description"]
        )
    )

    tags_train = tfidf_tags.fit_transform(
        df.loc[train_idx, "hashtags"]
    )


    # --------------------------------------------------------
    # NUMERIC FEATURES
    # --------------------------------------------------------

    numeric_columns = [
        "title_length",
        "description_length",
        "description_word_count",
        "tag_count",
        "has_description",
        "has_tags",
        "title_has_question"
    ]

    numeric_train = csr_matrix(
        df.loc[
            train_idx,
            numeric_columns
        ].values
    )


    # --------------------------------------------------------
    # COMBINE FEATURES
    # --------------------------------------------------------

    X_train = hstack([
        title_train,
        description_train,
        tags_train,
        numeric_train
    ]).tocsr()

    y_train = df.loc[
        train_idx,
        "Trending"
    ]


    print(
        "Training features:",
        X_train.shape
    )


    # --------------------------------------------------------
    # TRAIN MODEL
    # --------------------------------------------------------

    print("\nTraining Logistic Regression...")

    best_model = LogisticRegression(
        max_iter=1000,
        random_state=42
    )

    best_model.fit(
        X_train,
        y_train
    )


    print("✅ Training completed!")


    # --------------------------------------------------------
    # SAVE EVERYTHING
    # --------------------------------------------------------

    joblib.dump(
        best_model,
        MODEL_PATH
    )

    joblib.dump(
        tfidf_title,
        TITLE_PATH
    )

    joblib.dump(
        tfidf_description,
        DESCRIPTION_PATH
    )

    joblib.dump(
        tfidf_tags,
        TAGS_PATH
    )


    print("\n✅ Model saved successfully!")

    print("Location:")
    print(MODEL_DIR)


# ============================================================
# FINAL CHECK
# ============================================================

print("\n" + "=" * 60)
print("TESTING ENVIRONMENT READY")
print("=" * 60)

print("Model:", type(best_model).__name__)
print("Model file:", MODEL_PATH)
print("TF-IDF files loaded.")
print("✅ Ready for prediction!")

Project: C:\Users\Rajas\Downloads\youtube-project
Dataset exists: False

✅ Existing model found.
Loading...
✅ Model loaded successfully!

TESTING ENVIRONMENT READY
Model: LogisticRegression
Model file: C:\Users\Rajas\Downloads\youtube-project\models\best_model.pkl
TF-IDF files loaded.
✅ Ready for prediction!


In [ ]:
def predict_video(title, description, hashtags):
    title = str(title or "")
    description = str(description or "")
    hashtags = str(hashtags or "")

    title_length = len(title)
    description_length = len(description)
    description_word_count = len(description.split())
    tag_count = len(hashtags.split())
    has_description = int(description.strip() != "")
    has_tags = int(hashtags.strip() != "")
    title_has_question = int("?" in title)

    title_features = tfidf_title.transform([title])
    description_features = tfidf_description.transform([description])
    tags_features = tfidf_tags.transform([hashtags])

    numeric_features = csr_matrix([[
        title_length,
        description_length,
        description_word_count,
        tag_count,
        has_description,
        has_tags,
        title_has_question
    ]])

    final_features = hstack([
        title_features,
        description_features,
        tags_features,
        numeric_features
    ]).tocsr()

    prediction = best_model.predict(final_features)[0]
    probability = best_model.predict_proba(final_features)[0][1]

    return prediction, probability

In [ ]:
sample_title = "Amazing New Smartphone Features You Must See"
sample_description = "A detailed look at the latest smartphone features, camera, battery and performance."
sample_hashtags = "#technology #smartphone #tech"

prediction, probability = predict_video(
    sample_title,
    sample_description,
    sample_hashtags
)

print("Prediction:", "High-View" if prediction == 1 else "Not High-View")
print(f"Probability: {probability:.2%}")

Prediction: Not High-View
Probability: 44.42%


In [ ]:
test_videos = [
    (
        "Amazing New Smartphone Features You Must See",
        "A detailed look at the latest smartphone features, camera and battery.",
        "#technology #smartphone #tech"
    ),
    (
        "My Daily Routine",
        "Just a normal day and some things I did today.",
        ""
    ),
    (
        "You Won't Believe What Happened!",
        "An unexpected event with a surprising ending.",
        "#viral #shorts #funny"
    )
]

for title, description, hashtags in test_videos:
    prediction, probability = predict_video(title, description, hashtags)

    print("\nTitle:", title)
    print("Prediction:", "High-View" if prediction == 1 else "Not High-View")
    print(f"Probability: {probability:.2%}")


Title: Amazing New Smartphone Features You Must See
Prediction: Not High-View
Probability: 40.73%

Title: My Daily Routine
Prediction: Not High-View
Probability: 16.01%

Title: You Won't Believe What Happened!
Prediction: Not High-View
Probability: 37.18%


In [ ]:
print("Model expects:", best_model.n_features_in_, "features")

print(
    "Our pipeline creates:",
    tfidf_title.transform(["test"]).shape[1]
    + tfidf_description.transform(["test"]).shape[1]
    + tfidf_tags.transform(["test"]).shape[1]
    + 7,
    "features"
)

Model expects: 13007 features
Our pipeline creates: 13007 features


In [1]:

import joblib
from pathlib import Path

MODEL_DIR = Path("../models")

X_test_final = joblib.load(MODEL_DIR / "X_test_final.pkl")
y_test = joblib.load(MODEL_DIR / "y_test.pkl")

print("Test data loaded successfully!")
print("Test features:", X_test_final.shape)
print("Test labels:", y_test.shape)

Test data loaded successfully!
Test features: (3518, 21007)
Test labels: (3518,)


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = best_model.predict(X_test_final)

print(classification_report(y_test, y_pred))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))



              precision    recall  f1-score   support

           0       0.93      0.85      0.89      3166
           1       0.24      0.41      0.30       352

    accuracy                           0.81      3518
   macro avg       0.58      0.63      0.59      3518
weighted avg       0.86      0.81      0.83      3518

Confusion matrix:
[[2698  468]
 [ 207  145]]


In [ ]:
import joblib

tfidf_title = joblib.load('tfidf_title.pkl')
tfidf_description = joblib.load('tfidf_description.pkl')
tfidf_tags = joblib.load('tfidf_tags.pkl')

print("All 3 TF-IDF vectorizers loaded successfully!")

All 3 TF-IDF vectorizers loaded successfully!


In [ ]:
print("Model loaded:", 'best_model' in globals())
print("Title vectorizer loaded:", 'tfidf_title' in globals())
print("Description vectorizer loaded:", 'tfidf_description' in globals())
print("Hashtags vectorizer loaded:", 'tfidf_tags' in globals())

Model loaded: True
Title vectorizer loaded: True
Description vectorizer loaded: True
Hashtags vectorizer loaded: True


In [ ]:
from scipy.sparse import csr_matrix, hstack

print("Sparse matrix tools loaded!")

Sparse matrix tools loaded!


In [ ]:

def predict_video(title, description, hashtags):
    title = str(title or "")
    description = str(description or "")
    hashtags = str(hashtags or "")

    title_features = tfidf_title.transform([title])
    description_features = tfidf_description.transform([description])
    tags_features = tfidf_tags.transform([hashtags])

    numeric_features = csr_matrix([[
        len(title),
        len(description),
        len(description.split()),
        len(hashtags.split()),
        int(bool(description.strip())),
        int(bool(hashtags.strip())),
        int("?" in title)
    ]])

    final_features = hstack([
        title_features,
        description_features,
        tags_features,
        numeric_features
    ]).tocsr()

    prediction = best_model.predict(final_features)[0]
    probability = best_model.predict_proba(final_features)[0][1]

    return prediction, probability

print("Prediction function is ready!")

Prediction function is ready!


In [ ]:

# ============================================================
# YOUTUBE HIGH-VIEW PREDICTOR - COMPLETE TESTING CELL
# ============================================================

import joblib
import numpy as np
from scipy.sparse import hstack, csr_matrix
from IPython.display import display, clear_output
import ipywidgets as widgets

# ------------------------------------------------------------
# 1. LOAD SAVED MODEL AND TF-IDF VECTORIZERS
# ------------------------------------------------------------

best_model = joblib.load("best_model.pkl")
tfidf_title = joblib.load("tfidf_title.pkl")
tfidf_description = joblib.load("tfidf_description.pkl")
tfidf_tags = joblib.load("tfidf_tags.pkl")

print("✅ Model and TF-IDF vectorizers loaded successfully!")

# ------------------------------------------------------------
# 2. PREDICTION FUNCTION
# ------------------------------------------------------------

def predict_video(title, description, hashtags):

    # Convert inputs to strings
    title = str(title or "")
    description = str(description or "")
    hashtags = str(hashtags or "")

    # Numeric features
    title_length = len(title)
    description_length = len(description)
    description_word_count = len(description.split())
    tag_count = len(hashtags.split())

    has_description = int(description.strip() != "")
    has_tags = int(hashtags.strip() != "")
    title_has_question = int("?" in title)

    # TF-IDF features
    title_features = tfidf_title.transform([title])
    description_features = tfidf_description.transform([description])
    tags_features = tfidf_tags.transform([hashtags])

    # Numeric feature matrix
    numeric_features = csr_matrix([[
        title_length,
        description_length,
        description_word_count,
        tag_count,
        has_description,
        has_tags,
        title_has_question
    ]])

    # Combine all features
    final_features = hstack([
        title_features,
        description_features,
        tags_features,
        numeric_features
    ]).tocsr()

    # Prediction
    prediction = best_model.predict(final_features)[0]
    probability = best_model.predict_proba(final_features)[0][1]

    return prediction, probability


# ------------------------------------------------------------
# 3. TEST THE FUNCTION
# ------------------------------------------------------------

test_prediction, test_probability = predict_video(
    "AI Smartphone Review 2026",
    "Detailed review of the latest smartphone features and performance.",
    "#smartphone #technology #ai"
)

print("✅ Prediction function working!")
print("Test prediction:", test_prediction)
print("Test probability:", f"{test_probability:.2%}")


# ------------------------------------------------------------
# 4. CREATE USER INTERFACE
# ------------------------------------------------------------

title_input = widgets.Text(
    description="Title:",
    placeholder="Enter your YouTube video title",
    layout=widgets.Layout(width="90%"),
    style={"description_width": "80px"}
)

description_input = widgets.Textarea(
    description="Description:",
    placeholder="Describe your video...",
    layout=widgets.Layout(width="90%", height="120px"),
    style={"description_width": "80px"}
)

hashtags_input = widgets.Text(
    description="Hashtags:",
    placeholder="#technology #ai #shorts",
    layout=widgets.Layout(width="90%"),
    style={"description_width": "80px"}
)

predict_button = widgets.Button(
    description="Predict Video Potential",
    button_style="primary",
    icon="check"
)

output = widgets.Output()


# ------------------------------------------------------------
# 5. BUTTON FUNCTION
# ------------------------------------------------------------

def on_predict_clicked(b):

    with output:
        clear_output()

        title = title_input.value.strip()
        description = description_input.value.strip()
        hashtags = hashtags_input.value.strip()

        if not title:
            print("⚠️ Please enter a video title.")
            return

        prediction, probability = predict_video(
            title,
            description,
            hashtags
        )

        print("=" * 50)
        print("       YOUTUBE HIGH-VIEW PREDICTION")
        print("=" * 50)
        print()
        print("Title:", title)
        print()
        print(
            "Result:",
            "HIGH-VIEW" if prediction == 1 else "NOT HIGH-VIEW"
        )
        print()
        print(f"Estimated probability: {probability:.1%}")
        print()
        print("⚠️ This is a model estimate, not a guarantee.")
        print("=" * 50)


# Connect button
predict_button.on_click(on_predict_clicked)


# ------------------------------------------------------------
# 6. DISPLAY INTERFACE
# ------------------------------------------------------------

display(
    widgets.VBox([
        widgets.HTML("<h2>📺 YouTube High-View Predictor</h2>"),
        widgets.HTML(
            "<p>Enter the video's title, description and hashtags "
            "to estimate whether it belongs to the dataset's "
            "high-view class.</p>"
        ),
        title_input,
        description_input,
        hashtags_input,
        predict_button,
        output
    ])
)

print("✅ Predictor interface is ready!")

✅ Model and TF-IDF vectorizers loaded successfully!
✅ Prediction function working!
Test prediction: 1
Test probability: 67.65%


✅ Predictor interface is ready!


In [ ]:

import joblib
from pathlib import Path

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

MODEL_DIR = Path("../models")

# Load the updated saved model
best_model = joblib.load(MODEL_DIR / "best_model.pkl")

# Make predictions on the saved test data
y_pred = best_model.predict(X_test_final)

# Calculate performance metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("=" * 60)
print("MODEL PERFORMANCE ON HELD-OUT TEST SET")
print("=" * 60)

print(f"Accuracy  : {accuracy:.4f} ({accuracy:.2%})")
print(f"Precision : {precision:.4f} ({precision:.2%})")
print(f"Recall    : {recall:.4f} ({recall:.2%})")
print(f"F1-Score  : {f1:.4f} ({f1:.2%})")

print("\nCLASSIFICATION REPORT")
print(classification_report(y_test, y_pred, zero_division=0))

print("CONFUSION MATRIX")
print(confusion_matrix(y_test, y_pred))

MODEL PERFORMANCE ON HELD-OUT TEST SET

Accuracy  : 0.8081  (80.81%)
Precision : 0.2365  (23.65%)
Recall    : 0.4119  (41.19%)
F1-Score  : 0.3005  (30.05%)

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.93      0.85      0.89      3166
           1       0.24      0.41      0.30       352

    accuracy                           0.81      3518
   macro avg       0.58      0.63      0.59      3518
weighted avg       0.86      0.81      0.83      3518

CONFUSION MATRIX
[[2698  468]
 [ 207  145]]


In [ ]:
X_test_final = joblib.load('X_test_final.pkl')
y_test = joblib.load('y_test.pkl')

print("Test data loaded successfully!")
print("Test features:", X_test_final.shape)
print("Test labels:", y_test.shape)

Test data loaded successfully!
Test features: (3518, 13007)
Test labels: (3518,)


In [4]:

from scipy.sparse import hstack, csr_matrix

def predict_video(title, description, hashtags):

    # Clean inputs
    title = str(title or "")
    description = str(description or "")
    hashtags = str(hashtags or "")

    # Check that the model and vectorizers are loaded
    required = [
        "best_model",
        "tfidf_title",
        "tfidf_description",
        "tfidf_tags"
    ]

    missing = [name for name in required if name not in globals()]
    if missing:
        raise NameError(
            "Missing model components: " + ", ".join(missing)
            + ". Run your model-loading or training cell first."
        )

    # Numeric features (must match the training pipeline)
    title_length = len(title)
    description_length = len(description)
    description_word_count = len(description.split())
    tag_count = len(hashtags.split())
    has_description = int(bool(description.strip()))
    has_tags = int(bool(hashtags.strip()))
    title_has_question = int("?" in title)

    # Transform text using the fitted vectorizers
    title_features = tfidf_title.transform([title])
    description_features = tfidf_description.transform([description])
    tags_features = tfidf_tags.transform([hashtags])

    # Create numeric feature matrix
    numeric_features = csr_matrix([[
        title_length,
        description_length,
        description_word_count,
        tag_count,
        has_description,
        has_tags,
        title_has_question
    ]])

    # Combine features in the same order as training
    final_features = hstack([
        title_features,
        description_features,
        tags_features,
        numeric_features
    ]).tocsr()

    # Predict class and probability
    prediction = best_model.predict(final_features)[0]
    probability = best_model.predict_proba(final_features)[0][1]

    return prediction, probability


print("✅ predict_video() is ready!")

✅ predict_video() is ready!


In [1]:

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ---------- DASHBOARD THEME ----------
display(HTML("""
<style>
.yt-dashboard {
    background: #101827;
    color: #f8fafc;
    padding: 24px;
    border-radius: 18px;
    font-family: Arial, sans-serif;
    border: 1px solid #263449;
    max-width: 1000px;
    margin: 10px auto;
}
.yt-title {
    font-size: 27px;
    font-weight: 800;
    color: #ffffff;
    margin-bottom: 5px;
}
.yt-subtitle {
    color: #9caec4;
    font-size: 13px;
    margin-bottom: 22px;
}
.yt-section {
    color: #e2e8f0;
    font-size: 15px;
    font-weight: 700;
    margin: 15px 0 8px;
}
.yt-result {
    background: #1b293b;
    border: 1px solid #33445b;
    padding: 18px;
    border-radius: 13px;
    margin-top: 16px;
}
.yt-muted {
    color: #9caec4;
    font-size: 12px;
}
.yt-big {
    font-size: 25px;
    font-weight: 800;
    margin: 8px 0;
}
.yt-bar-bg {
    background: #334155;
    height: 12px;
    border-radius: 20px;
    overflow: hidden;
    margin: 10px 0;
}
.yt-bar {
    height: 12px;
    border-radius: 20px;
}
.yt-warning {
    color: #fbbf24;
    font-size: 12px;
    margin-top: 14px;
    line-height: 1.5;
}
</style>
"""))

# ---------- INPUT FIELDS ----------
title_input = widgets.Textarea(
    value="",
    placeholder="Enter your YouTube video title...",
    description="",
    layout=widgets.Layout(width="100%", height="75px")
)

description_input = widgets.Textarea(
    value="",
    placeholder="Enter the video description...",
    description="",
    layout=widgets.Layout(width="100%", height="120px")
)

hashtags_input = widgets.Textarea(
    value="",
    placeholder="Example: #technology #AI #tutorial",
    description="",
    layout=widgets.Layout(width="100%", height="65px")
)

# ---------- BUTTONS ----------
predict_button = widgets.Button(
    description="Predict performance",
    icon="bar-chart",
    button_style="danger",
    layout=widgets.Layout(width="220px", height="42px")
)

sample_button = widgets.Button(
    description="Load sample",
    icon="flask",
    layout=widgets.Layout(width="145px", height="42px")
)

clear_button = widgets.Button(
    description="Clear fields",
    icon="trash",
    layout=widgets.Layout(width="130px", height="42px")
)

# ---------- OUTPUT AREA ----------
result_output = widgets.Output(
    layout=widgets.Layout(width="100%")
)

# ---------- DASHBOARD LAYOUT ----------
header = widgets.HTML("""
<div class="yt-title">▶ YouTube Analytics Studio</div>
<div class="yt-subtitle">
    Machine Learning • Video Performance Prediction • Logistic Regression
</div>
""")

input_heading = widgets.HTML(
    "<div class='yt-section'>VIDEO DETAILS</div>"
)

title_heading = widgets.HTML(
    "<div style='color:#cbd5e1;font-weight:600;margin:5px 0'>Video title</div>"
)
description_heading = widgets.HTML(
    "<div style='color:#cbd5e1;font-weight:600;margin:5px 0'>Description</div>"
)
hashtags_heading = widgets.HTML(
    "<div style='color:#cbd5e1;font-weight:600;margin:5px 0'>Hashtags / tags</div>"
)

buttons = widgets.HBox(
    [predict_button, sample_button, clear_button],
    layout=widgets.Layout(
        display="flex",
        flex_flow="row wrap",
        gap="10px",
        margin="18px 0 5px"
    )
)

dashboard = widgets.VBox([
    header,
    input_heading,
    title_heading,
    title_input,
    description_heading,
    description_input,
    hashtags_heading,
    hashtags_input,
    buttons,
    result_output
], layout=widgets.Layout(
    width="100%",
    max_width="1000px",
    padding="20px",
    border="1px solid #263449"
))

# ---------- PREDICTION ----------
def run_prediction(button):
    with result_output:
        clear_output(wait=True)

        title = title_input.value.strip()
        description = description_input.value.strip()
        hashtags = hashtags_input.value.strip()

        if not title:
            display(HTML("""
            <div class="yt-result" style="border-color:#ef4444">
                <b style="color:#fca5a5">Please enter a video title first.</b>
            </div>
            """))
            return

        try:
            prediction, probability = predict_video(
                title, description, hashtags
            )

            probability = float(probability)
            percentage = max(0, min(100, probability * 100))

            if int(prediction) == 1:
                label = "HIGH-VIEW"
                color = "#22c55e"
                message = "The model predicts this video belongs to the high-view class."
            else:
                label = "NOT HIGH-VIEW"
                color = "#f97316"
                message = "The model predicts this video belongs to the other class."

            display(HTML(f"""
            <div class="yt-result">
                <div class="yt-muted">MODEL PREDICTION</div>
                <div class="yt-big" style="color:{color}">
                    {label}
                </div>
                <div style="color:#cbd5e1;font-size:14px">
                    {message}
                </div>

                <div style="margin-top:20px;color:#e2e8f0;font-weight:700">
                    Estimated probability of HIGH-VIEW class
                </div>
                <div class="yt-bar-bg">
                    <div class="yt-bar"
                         style="width:{percentage:.1f}%;background:{color}">
                    </div>
                </div>
                <div style="font-size:20px;font-weight:800;color:#ffffff">
                    {percentage:.1f}%
                </div>
                <div class="yt-muted">
                    Model probability for class 1, not a guarantee of actual views.
                </div>
            </div>

            <div class="yt-result">
                <div style="font-weight:700;color:#e2e8f0;margin-bottom:10px">
                    VIDEO SUMMARY
                </div>
                <div style="color:#cbd5e1;line-height:1.8">
                    <b>Title:</b> {title[:250]}<br>
                    <b>Title length:</b> {len(title)} characters<br>
                    <b>Description:</b> {len(description)} characters<br>
                    <b>Tags field:</b> {"Provided" if hashtags else "Empty"}
                </div>
            </div>

            <div class="yt-warning">
                ⚠️ This is a dataset-defined top-10%-by-views proxy,
                not YouTube's official Trending system. The probability
                is a model output, not a measured chance of going viral.
            </div>
            """))

        except Exception as e:
            display(HTML(
                "<div class='yt-result' style='color:#fca5a5'>"
                "<b>Prediction error</b><br>"
                + str(e).replace("&", "&amp;").replace("<", "&lt;")
                + "</div>"
            ))

# ---------- SAMPLE / CLEAR ----------
def load_sample(button):
    title_input.value = "Amazing AI Tools You Need to Try in 2026!"
    description_input.value = (
        "Discover useful artificial intelligence tools that can "
        "help with productivity, creativity, and everyday work. "
        "In this video, we explore practical examples and tips."
    )
    hashtags_input.value = "#AI #technology #productivity #tutorial"

def clear_fields(button):
    title_input.value = ""
    description_input.value = ""
    hashtags_input.value = ""
    with result_output:
        clear_output()

predict_button.on_click(run_prediction)
sample_button.on_click(load_sample)
clear_button.on_click(clear_fields)

display(dashboard)